In [1]:
# -*- coding: utf-8 -*-
"""
텍스트 토픽모델링(LDA) 기반 감성 해석 파이프라인
====================================================
목적: 리뷰 문장의 토픽(주제)이 감성(긍정/부정 강도)에 어떤 영향을 미치는지
      토픽모델링 + 회귀 계수로 해석하고, 시각화로 보여준다.

파이프라인 순서:
  1. 파일 로드 + 기본 노이즈 정제
  2. VADER로 문장별 감성 점수 계산 (원문 기준, 정제 전!)
  3. 감성어를 제외한 LDA용 텍스트 정제
  4. CountVectorizer + LDA로 토픽 추출
  5. 토픽별 top 단어 확인 (사람이 라벨링)
  6. Ridge 회귀로 "토픽 -> 감성" 계수 추정
  7. 시각화: 토픽별 평균 감성, 회귀계수, 토픽x제품 히트맵, 대표 문장

사전 설치 필요 (최초 1회):
    pip install nltk scikit-learn pandas matplotlib seaborn
    python -c "import nltk; nltk.download('vader_lexicon'); nltk.download('stopwords'); nltk.download('wordnet'); nltk.download('omw-1.4')"
"""

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score



In [11]:
# 데이터불러오기
DATA_DIR = "../data/topics"     # .data 파일들이 있는 폴더 경로
OUTPUT_DIR = "../data/process"  # 결과물(csv, png) 저장 폴더
N_TOPICS = 12                          # 해석 가능한 수준으로 고정 (10~15 권장)
MAX_FEATURES = 3000                    # CountVectorizer 어휘 크기
RANDOM_STATE = 42
 
plt.rcParams['font.family'] = 'AppleGothic'  # 한글 폰트 (Windows는 'Malgun Gothic'으로 변경)
plt.rcParams['axes.unicode_minus'] = False
 

In [12]:
# =========================================================
# 1. 파일 로드 + 기본 노이즈 정제
# =========================================================
def load_raw_data(data_dir: str) -> pd.DataFrame:
    """51개 {aspect}_{product}_txt.data 파일을 읽어 문장 단위 DataFrame으로 변환."""
    files = glob.glob(os.path.join(data_dir, "*.data"))
    if not files:
        raise FileNotFoundError(f"{data_dir} 에서 .data 파일을 찾지 못했습니다.")
 
    records = []
    for fpath in files:
        fname = os.path.basename(fpath).replace("_txt.data", "").replace(".data", "")
        parts = fname.split("_")
        aspect = parts[0]
        product = "_".join(parts[1:]) if len(parts) > 1 else "unknown"
 
        with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = basic_clean(line)
                if line:
                    records.append({"aspect": aspect, "product": product, "raw_text": line})
 
    df = pd.DataFrame(records)
    print(f"[로드 완료] 총 {len(df)}개 문장, {df['product'].nunique()}개 제품, {df['aspect'].nunique()}개 속성")
    return df

def basic_clean(text: str) -> str:
    """VADER 적용 전 최소한의 노이즈 정제 (원문의 대소문자/구두점은 보존)."""
    text = text.strip()
    text = text.replace("\r", "").replace("\n", " ")
    text = re.sub(r"\s*,\s*(?=\d)", "", text)   # "3, Cell" -> "3Cell" 같은 깨진 쉼표 정리
    text = re.sub(r"\s{2,}", " ", text)          # 중복 공백 정리
    text = text.strip(" .,")
    return text
 
 

In [13]:
# =========================================================
# 2. VADER 감성 점수 계산 (원문 기준)
# =========================================================
def add_sentiment_scores(df: pd.DataFrame) -> pd.DataFrame:
    sia = SentimentIntensityAnalyzer()
    df["sentiment"] = df["raw_text"].apply(lambda x: sia.polarity_scores(x)["compound"])
 
    # 검증: 극단 문장 몇 개 눈으로 확인
    print("\n[VADER 검증] 가장 긍정적인 문장 3개")
    print(df.nlargest(3, "sentiment")[["raw_text", "sentiment"]].to_string(index=False))
    print("\n[VADER 검증] 가장 부정적인 문장 3개")
    print(df.nsmallest(3, "sentiment")[["raw_text", "sentiment"]].to_string(index=False))
 
    return df, sia
 
 

In [14]:
# =========================================================
# 3. 감성어를 제외한 LDA용 정제
# =========================================================
def build_sentiment_word_set(sia: SentimentIntensityAnalyzer) -> set:
    """VADER lexicon에서 감성 강도가 있는 단어를 뽑아 stopword처럼 사용."""
    sentiment_words = {w.lower() for w, score in sia.lexicon.items() if abs(score) >= 1.0}
    return sentiment_words
 
 
def clean_for_topic(text: str, stop_words: set, sentiment_words: set,
                     lemmatizer: WordNetLemmatizer) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t) for t in tokens
        if t not in stop_words
        and t not in sentiment_words   # 감성어 제외 -> 토픽이 '주제'만 담도록
        and len(t) > 2
    ]
    return " ".join(tokens)
 
 

In [15]:
#  =========================================================
# 4. CountVectorizer + LDA
# =========================================================
def run_lda(df: pd.DataFrame, n_topics: int, max_features: int):
    vectorizer = CountVectorizer(
        max_features=max_features,
        min_df=3,
        max_df=0.9,
        ngram_range=(1, 2),
    )
    doc_term_matrix = vectorizer.fit_transform(df["clean_text"])
 
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        learning_method="online",
        max_iter=15,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    topic_dist = lda.fit_transform(doc_term_matrix)
 
    topic_cols = [f"topic_{i}" for i in range(n_topics)]
    topic_df = pd.DataFrame(topic_dist, columns=topic_cols, index=df.index)
 
    return vectorizer, lda, topic_df, topic_cols
 
 
def print_top_words(lda: LatentDirichletAllocation, vectorizer: CountVectorizer, n_top=10):
    feature_names = vectorizer.get_feature_names_out()
    topic_labels = {}
    print("\n[토픽별 대표 단어]")
    for idx, topic in enumerate(lda.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top - 1:-1]]
        print(f"  topic_{idx}: {', '.join(top_words)}")
        topic_labels[f"topic_{idx}"] = ", ".join(top_words[:3])  # 상위 3단어로 임시 라벨
    return topic_labels
 
 

In [16]:
# =========================================================
# 5. 회귀 (Ridge) - 토픽 -> 감성 해석
# =========================================================
def run_regression(df: pd.DataFrame, topic_df: pd.DataFrame, topic_cols: list):
    product_dummies = pd.get_dummies(df["product"], prefix="product")
    aspect_dummies = pd.get_dummies(df["aspect"], prefix="aspect")
 
    X = pd.concat([topic_df, product_dummies, aspect_dummies], axis=1)
    y = df["sentiment"]
 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
 
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
 
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"\n[회귀 결과] RMSE={rmse:.4f}, R2={r2:.4f}  (참고용 지표, 목적은 해석)")
 
    # 토픽 계수만 추출 (해석의 핵심)
    coef_series = pd.Series(model.coef_, index=X.columns)
    topic_coef = coef_series[topic_cols].sort_values()
 
    return model, topic_coef
 
 

In [17]:
# =========================================================
# 6. 시각화
# =========================================================
def visualize(df, topic_df, topic_cols, topic_labels, topic_coef):
    df_full = pd.concat([df.reset_index(drop=True), topic_df.reset_index(drop=True)], axis=1)
 
    # (1) 토픽별 회귀계수 (감성에 미치는 영향력, 방향)
    plt.figure(figsize=(9, 6))
    labels = [f"{t}\n({topic_labels.get(t,'')})" for t in topic_coef.index]
    colors = ["#d62728" if v < 0 else "#2ca02c" for v in topic_coef.values]
    plt.barh(labels, topic_coef.values, color=colors)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title("토픽별 감성 영향력 (Ridge 회귀계수)")
    plt.xlabel("계수 (감성 점수에 대한 영향)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "topic_coefficients.png"), dpi=150)
    plt.close()
 
    # (2) 토픽별 평균 감성 점수
    dominant_topic = topic_df[topic_cols].idxmax(axis=1)
    df_full["dominant_topic"] = dominant_topic
    topic_sentiment = df_full.groupby("dominant_topic")["sentiment"].mean().reindex(topic_cols)
 
    plt.figure(figsize=(9, 6))
    colors2 = ["#d62728" if v < 0 else "#2ca02c" for v in topic_sentiment.values]
    plt.barh(topic_sentiment.index, topic_sentiment.values, color=colors2)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title("토픽별 평균 감성 점수 (해당 토픽이 지배적인 문장 기준)")
    plt.xlabel("평균 VADER 감성 점수")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "topic_mean_sentiment.png"), dpi=150)
    plt.close()
 
    # (3) 토픽 x 제품 히트맵 (평균 감성)
    pivot = df_full.pivot_table(
        index="product", columns="dominant_topic", values="sentiment", aggfunc="mean"
    ).reindex(columns=topic_cols)
 
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot, cmap="RdYlGn", center=0, annot=False, linewidths=0.3)
    plt.title("제품 x 토픽 평균 감성 히트맵")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "product_topic_heatmap.png"), dpi=150)
    plt.close()
 
    print(f"\n[시각화 저장 완료] {OUTPUT_DIR}/ 에 png 3개 저장됨")
    return df_full
 

In [18]:

 
# =========================================================
# 실행
# =========================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)  # data/process 폴더 없으면 생성
 
    # 1. 로드
    df = load_raw_data(DATA_DIR)
 
    # 2. VADER 감성 점수
    df, sia = add_sentiment_scores(df)
 
    # 3. 감성어 제외 정제
    stop_words = set(stopwords.words("english"))
    sentiment_words = build_sentiment_word_set(sia)
    lemmatizer = WordNetLemmatizer()
 
    df["clean_text"] = df["raw_text"].apply(
        lambda x: clean_for_topic(x, stop_words, sentiment_words, lemmatizer)
    )
    df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)
 
    # 4. LDA
    vectorizer, lda, topic_df, topic_cols = run_lda(df, N_TOPICS, MAX_FEATURES)
    topic_labels = print_top_words(lda, vectorizer)
 
    # 5. 회귀 (해석용)
    model, topic_coef = run_regression(df, topic_df, topic_cols)
 
    print("\n[감성을 가장 높이는 토픽 TOP3]")
    print(topic_coef.sort_values(ascending=False).head(3))
    print("\n[감성을 가장 낮추는 토픽 TOP3]")
    print(topic_coef.sort_values().head(3))
 
    # 6. 시각화
    df_full = visualize(df, topic_df, topic_cols, topic_labels, topic_coef)
 
    # 결과 저장
    result_path = os.path.join(OUTPUT_DIR, "topic_sentiment_result.csv")
    df_full.to_csv(result_path, index=False, encoding="utf-8-sig")
    print(f"\n[전체 완료] 결과 CSV 저장: {result_path}")
 

[로드 완료] 총 7086개 문장, 13개 제품, 36개 속성

[VADER 검증] 가장 긍정적인 문장 3개
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      raw_text  sentiment
my first fill up was 26 mpg mixed city and hwy I only expect it to get better, steering is tight and precise, only complaints are road noise is more than i like but its livable, rain or just dew pours in right on top of the power window controls when the window is cracked, but window guards have fixed that, its a fun car to drive, and for what it is, its comfortable, controls are great, easy to reach, I'm looking forward to a lot of great miles with this car   

findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
C:\Users\mega\AppData\Local\Temp\ipykernel_12428\4051747878.py:15: UserWarning: Glyph 44228 (\N{HANGUL SYLLABLE GYE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\mega\AppData\Local\


[토픽별 대표 단어]
  topic_0: even, feel, read, eye, though, seems, quot, review, reading, internet
  topic_1: button, front, work, page, however, stay, front desk, high, hard, made
  topic_2: two, size, minute, everything, time, using, first, could, font, walk
  topic_3: screen, battery, video, life, battery life, much, use, size, sound, keyboard
  topic_4: price, performance, faster, many, issue, think, smaller, overall, light, looking
  topic_5: little, tube, road, make, station, city, gloucester, camry, convenient, noise
  topic_6: room, get, voice, bathroom, speed, really, desk, bed, day, small
  topic_7: location, would, new, take, right, turn, wharf, area, direction, fisherman
  topic_8: staff, room, lot, small, room small, smooth, booked, rate, quickly, walking
  topic_9: interior, need, seat, room, standard, check, enough, big, satellite, coffee
  topic_10: hotel, service, room, staff, location, transmission, food, room service, night, hotel location
  topic_11: mileage, car, gas, g

findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Font family 'AppleGothic' not found.
findfont: Fon


[시각화 저장 완료] ../data/process/ 에 png 3개 저장됨

[전체 완료] 결과 CSV 저장: ../data/process\topic_sentiment_result.csv
